In [1]:
%pip install pandas numpy scikit-learn transformers torch sentence-transformers matplotlib tqdm

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import re
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt

from transformers import pipeline, AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

RANDOM_STATE = 42
from tqdm.auto import tqdm

/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Change this path if necessary
df = pd.read_csv("/Users/yume/PycharmProjects/PBL2/data/reddit_dr.csv")

print(df.shape)
print(df.columns)

(12854, 9)
Index(['Title', 'Political Lean', 'Score', 'Id', 'Subreddit', 'URL',
       'Num of Comments', 'Text', 'Date Created'],
      dtype='object')


In [3]:
df["Title"] = df["Title"].fillna("").astype(str).str.strip()
df["Text"] = df["Text"].fillna("").astype(str).str.strip()

df["Text"] = (
    df["Title"]
    + " "
    + df["Text"]
).str.strip()

df["Political Lean"] = (
    df["Political Lean"]
    .astype(str)
    .str.strip()
)

df = df[
    df["Political Lean"].isin(
        ["Conservative", "Liberal"]
    )
].copy()

df = df[
    df["Text"].str.len() > 0
].reset_index(drop=True)

## RoBERTa sentiment features

In [4]:
sentiment_model = pipeline(
    task="text-classification",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    top_k=None,
    device=-1
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 37989.14it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
texts = df["Text"].tolist()

sentiment_results = sentiment_model(
    texts,
    batch_size=16,
    truncation=True,
    max_length=512
)

In [6]:
sentiment_rows = []

for result in sentiment_results:
    score_dictionary = {
        item["label"].lower(): item["score"]
        for item in result
    }

    # Handle models that return LABEL_0, LABEL_1, LABEL_2
    if "label_0" in score_dictionary:
        negative_score = score_dictionary.get("label_0", 0.0)
        neutral_score = score_dictionary.get("label_1", 0.0)
        positive_score = score_dictionary.get("label_2", 0.0)
    else:
        negative_score = score_dictionary.get("negative", 0.0)
        neutral_score = score_dictionary.get("neutral", 0.0)
        positive_score = score_dictionary.get("positive", 0.0)

    sentiment_rows.append({
        "roberta_negative": negative_score,
        "roberta_neutral": neutral_score,
        "roberta_positive": positive_score
    })

sentiment_df = pd.DataFrame(sentiment_rows)

sentiment_df.head()

,roberta_negative,roberta_neutral,roberta_positive
0,0.071206,0.681420,0.247375
1,0.004685,0.463182,0.532133
2,0.022105,0.912309,0.065586
3,0.714801,0.263404,0.021795
4,0.128113,0.596820,0.275068


In [7]:
df[
    [
        "roberta_negative",
        "roberta_neutral",
        "roberta_positive"
    ]
] = sentiment_df[
    [
        "roberta_negative",
        "roberta_neutral",
        "roberta_positive"
    ]
]

## RoBERTa final hidden-layer embeddings

In [8]:
# This extracts one 768-dimensional contextual embedding per Reddit post.
# It uses the final hidden layer from roberta-base and mean-pools non-padding tokens.

embedding_model_name = "roberta-base"

embedding_tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)
embedding_model = AutoModel.from_pretrained(embedding_model_name)

if torch.cuda.is_available():
    embedding_device = torch.device("cuda")
elif torch.backends.mps.is_available():
    embedding_device = torch.device("mps")
else:
    embedding_device = torch.device("cpu")

embedding_model.to(embedding_device)
embedding_model.eval()

print("RoBERTa embedding device:", embedding_device)


def get_roberta_embeddings(
    texts,
    batch_size=16,
    max_length=256,
    normalize=True
):
    """
    Return a matrix of mean-pooled final-hidden-layer RoBERTa embeddings.

    Each row corresponds to one document. Padding tokens are excluded from
    the mean. L2 normalization is optional and enabled by default.
    """
    all_embeddings = []

    for start in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[start:start + batch_size]

        encoded = embedding_tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        encoded = {
            key: value.to(embedding_device)
            for key, value in encoded.items()
        }

        with torch.inference_mode():
            outputs = embedding_model(**encoded)

        last_hidden_state = outputs.last_hidden_state
        attention_mask = encoded["attention_mask"].unsqueeze(-1)

        masked_hidden_state = last_hidden_state * attention_mask
        summed_hidden_state = masked_hidden_state.sum(dim=1)
        token_counts = attention_mask.sum(dim=1).clamp(min=1)

        batch_embeddings = summed_hidden_state / token_counts

        if normalize:
            batch_embeddings = torch.nn.functional.normalize(
                batch_embeddings,
                p=2,
                dim=1
            )

        all_embeddings.append(
            batch_embeddings.cpu().numpy()
        )

    return np.vstack(all_embeddings)


roberta_embeddings = get_roberta_embeddings(
    df["Text"].tolist(),
    batch_size=16,
    max_length=256,
    normalize=True
)

print("RoBERTa embedding matrix shape:", roberta_embeddings.shape)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 16073.88it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RoBERTa embedding device: mps


100%|██████████| 804/804 [03:30<00:00,  3.81it/s]

RoBERTa embedding matrix shape: (12854, 768)


In [9]:
roberta_embedding_df = pd.DataFrame(
    roberta_embeddings,
    columns=[
        f"roberta_embed_{i}"
        for i in range(roberta_embeddings.shape[1])
    ],
    index=df.index
)

df = pd.concat(
    [df, roberta_embedding_df],
    axis=1
)

roberta_embedding_features = roberta_embedding_df.columns.tolist()

print("Added embedding columns:", len(roberta_embedding_features))
print("Any missing embedding values:",
      df[roberta_embedding_features].isna().any().any())

df[roberta_embedding_features].head()


Added embedding columns: 768
Any missing embedding values: False


,roberta_embed_0,roberta_embed_1,roberta_embed_2,roberta_embed_3,roberta_embed_4,roberta_embed_5,roberta_embed_6,roberta_embed_7,roberta_embed_8,roberta_embed_9,...,roberta_embed_758,roberta_embed_759,roberta_embed_760,roberta_embed_761,roberta_embed_762,roberta_embed_763,roberta_embed_764,roberta_embed_765,roberta_embed_766,roberta_embed_767
0,-0.003242,0.006471,0.008163,-0.018306,0.000701,-0.007300,0.002353,0.007196,0.005535,0.006050,...,-0.006963,0.001248,-0.000153,-0.000525,0.002770,0.016060,0.006154,0.008525,-0.001826,0.005844
1,0.001243,0.016864,-0.000646,-0.000353,-0.011800,0.003662,0.003680,-0.003607,-0.002468,-0.005838,...,0.009292,0.001557,-0.004341,0.000472,-0.013887,0.001274,0.025650,0.015789,0.014663,0.002135
2,0.003174,0.001009,0.001541,-0.004856,-0.000624,0.005158,0.004913,0.004936,0.004388,-0.000182,...,-0.011704,-0.005695,-0.018339,0.007639,0.006934,0.007147,0.038500,0.005797,0.004929,0.008291
3,0.000058,0.007821,0.001372,-0.019020,0.014534,0.020016,0.003161,-0.009625,0.023772,-0.001063,...,0.000584,0.001499,0.004944,-0.009547,0.016795,0.006821,0.012308,0.021465,-0.002174,0.008404
4,0.001112,0.006951,0.003057,-0.000319,0.009763,-0.005203,-0.000072,-0.005595,0.009163,-0.005824,...,-0.002086,-0.001511,-0.009190,-0.002525,-0.003701,0.009884,0.022952,0.024720,0.001317,0.006085


## Moral-foundation semantic features


In [16]:
moral_prototypes = {
    "care": [
        "The text emphasizes compassion, empathy, protection, and helping vulnerable people.",
        "The text supports reducing suffering and caring for people in need."
    ],
    "harm": [
        "The text describes cruelty, injury, neglect, violence, or causing suffering.",
        "The text condemns actions that hurt or endanger others."
    ],
    "fairness": [
        "The text emphasizes justice, equality, impartiality, and equal treatment.",
        "The text supports fair rules, equal rights, and equitable outcomes."
    ],
    "cheating": [
        "The text describes corruption, exploitation, dishonesty, or unfair advantage.",
        "The text accuses people or institutions of cheating or violating fair rules."
    ],
    "loyalty": [
        "The text emphasizes patriotism, group solidarity, loyalty, and commitment to a community.",
        "The text praises devotion to one's nation, party, or social group."
    ],
    "betrayal": [
        "The text describes betrayal, disloyalty, treason, or abandoning one's group.",
        "The text criticizes someone for failing their community, country, or allies."
    ],
    "authority": [
        "The text supports authority, hierarchy, obedience, law, and social order.",
        "The text expresses respect for leaders, institutions, rules, or tradition."
    ],
    "subversion": [
        "The text challenges authority, hierarchy, leaders, or established institutions.",
        "The text portrays authority or institutions as illegitimate, corrupt, or oppressive."
    ],
    "purity": [
        "The text emphasizes sanctity, purity, decency, cleanliness, or moral virtue.",
        "The text praises behavior seen as wholesome, sacred, or morally proper."
    ],
    "degradation": [
        "The text describes contamination, corruption, disgust, indecency, or moral degradation.",
        "The text portrays behavior or society as filthy, corrupt, or morally fallen."
    ],
    "liberty": [
        "The text emphasizes freedom, autonomy, civil rights, personal choice, and independence.",
        "The text supports resistance to coercion and protection of individual freedom."
    ],
    "oppression": [
        "The text describes domination, coercion, censorship, control, or loss of freedom.",
        "The text portrays people as being oppressed, restricted, or forced by powerful institutions."
    ]
}

In [11]:
semantic_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7344.04it/s]


In [17]:
prototype_names = list(moral_prototypes.keys())

prototype_embeddings = []

for name in prototype_names:
    embeddings = semantic_model.encode(
        moral_prototypes[name],
        normalize_embeddings=True
    )

    mean_embedding = embeddings.mean(axis=0)
    mean_embedding = mean_embedding / np.linalg.norm(mean_embedding)

    prototype_embeddings.append(mean_embedding)

prototype_embeddings = np.vstack(prototype_embeddings)

In [13]:
moral_scores = post_embeddings @ prototype_embeddings.T

moral_df = pd.DataFrame(
    moral_scores,
    columns=[f"moral_{name}" for name in prototype_names]
)

df[moral_df.columns] = moral_df

Batches: 100%|██████████| 402/402 [00:18<00:00, 21.75it/s]

(12854, 384)


In [14]:
df["care_minus_harm"] = df["moral_care"] - df["moral_harm"]
df["fairness_minus_cheating"] = (
    df["moral_fairness"] - df["moral_cheating"]
)
df["loyalty_minus_betrayal"] = (
    df["moral_loyalty"] - df["moral_betrayal"]
)
df["authority_minus_subversion"] = (
    df["moral_authority"] - df["moral_subversion"]
)
df["purity_minus_degradation"] = (
    df["moral_purity"] - df["moral_degradation"]
)
df["liberty_minus_oppression"] = (
    df["moral_liberty"] - df["moral_oppression"]
)

/var/folders/1z/hv4_qzyn6556dk5hqb4k6tdh0000gn/T/ipykernel_44560/211583188.py:2: RuntimeWarning: divide by zero encountered in matmul
  post_embeddings
/var/folders/1z/hv4_qzyn6556dk5hqb4k6tdh0000gn/T/ipykernel_44560/211583188.py:2: RuntimeWarning: overflow encountered in matmul
  post_embeddings
/var/folders/1z/hv4_qzyn6556dk5hqb4k6tdh0000gn/T/ipykernel_44560/211583188.py:2: RuntimeWarning: invalid value encountered in matmul
  post_embeddings


,moral_care,moral_fairness,moral_loyalty,moral_authority,moral_purity,moral_liberty
0,0.234892,0.314288,0.317456,0.310486,0.260284,0.282486
1,0.003793,-0.005315,0.003659,-0.095113,-0.050666,-0.008426
2,0.008286,0.072012,0.113781,0.017323,0.019100,0.075828
3,0.180345,0.158497,0.098507,0.064010,0.123075,0.065221
4,0.083044,0.031898,0.039451,0.001987,0.114041,0.032050


In [ ]:
df[
    moral_df.columns
] = moral_df

In [ ]:
moral_feature_names = [
    f"moral_{name}"
    for name in moral_names
]

df[moral_feature_names].describe()

In [ ]:
df.to_csv(
    "reddit_with_semantic_features.csv",
    index=False
)

print("Saved enriched dataframe.")

## Model feature groups

The models below use the same train/test split:

- **TF-IDF only**
- **RoBERTa sentiment + moral-foundation scores + all 768 RoBERTa embedding dimensions**
- **TF-IDF + all semantic features**

The 768 embedding dimensions are included as numeric predictors. They are not individually interpretable in the same way as TF-IDF terms or moral-foundation scores.


In [ ]:
sentiment_features = [
    "roberta_negative",
    "roberta_neutral",
    "roberta_positive"
]

moral_features = [
    "moral_care",
    "moral_fairness",
    "moral_loyalty",
    "moral_authority",
    "moral_purity",
    "moral_liberty"
]

# New: 768-dimensional final hidden-layer RoBERTa embeddings
# These capture broader semantic/contextual information than the 3 sentiment scores.
roberta_embedding_features = [
    column
    for column in df.columns
    if column.startswith("roberta_embed_")
]

semantic_features = (
    sentiment_features
    + moral_features
    + roberta_embedding_features
)

all_feature_columns = (
    ["Text"]
    + semantic_features
)

print("Number of sentiment features:", len(sentiment_features))
print("Number of moral features:", len(moral_features))
print("Number of RoBERTa embedding features:", len(roberta_embedding_features))
print("Total numeric semantic features:", len(semantic_features))
assert len(roberta_embedding_features) == 768, "Expected 768 RoBERTa embedding dimensions."


In [ ]:
model_df = df.dropna(
    subset=[
        "Text",
        "Political Lean"
    ] + semantic_features
).copy()

In [ ]:
X = model_df[all_feature_columns]
y = model_df["Political Lean"]

print(X.shape)
print(y.value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

In [ ]:
full_preprocessor = ColumnTransformer(
    transformers=[
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=3,
                max_df=0.95,
                sublinear_tf=True,
                max_features=50000
            ),
            "Text"
        ),

        (
            "semantic",
            StandardScaler(),
            semantic_features
        )
    ],
    remainder="drop"
)

In [ ]:
full_pipeline = Pipeline([
    (
        "preprocessor",
        full_preprocessor
    ),

    (
        "classifier",
        LogisticRegression(
            solver="liblinear",
            penalty="l2",
            class_weight="balanced",
            max_iter=3000,
            random_state=RANDOM_STATE
        )
    )
])

In [ ]:
full_param_grid = {
    "preprocessor__tfidf__ngram_range": [
        (1, 1),
        (1, 2)
    ],

    "preprocessor__tfidf__min_df": [
        2,
        3,
        5
    ],

    "preprocessor__tfidf__stop_words": [
        None,
        "english"
    ],

    "classifier__C": [
        0.1,
        0.5,
        1,
        2,
        5
    ]
}

In [ ]:
full_search = GridSearchCV(
    estimator=full_pipeline,
    param_grid=full_param_grid,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
    verbose=2,
    return_train_score=True
)

full_search.fit(
    X_train,
    y_train
)

In [ ]:
best_full_model = full_search.best_estimator_

full_predictions = best_full_model.predict(
    X_test
)

full_accuracy = accuracy_score(
    y_test,
    full_predictions
)

full_macro_f1 = f1_score(
    y_test,
    full_predictions,
    average="macro"
)

print("Best parameters:")
print(full_search.best_params_)

print("\nBest CV macro F1:")
print(full_search.best_score_)

print("\nTest accuracy:")
print(full_accuracy)

print("\nTest macro F1:")
print(full_macro_f1)

print("\nClassification report:")
print(
    classification_report(
        y_test,
        full_predictions
    )
)

print("\nConfusion matrix:")
print(
    confusion_matrix(
        y_test,
        full_predictions
    )
)

## TF-IDF only

In [ ]:
tfidf_only_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            sublinear_tf=True,
            max_df=0.95,
            max_features=50000
        )
    ),

    (
        "classifier",
        LogisticRegression(
            solver="liblinear",
            penalty="l2",
            class_weight="balanced",
            max_iter=3000,
            random_state=RANDOM_STATE
        )
    )
])

In [ ]:
tfidf_only_param_grid = {
    "tfidf__ngram_range": [
        (1, 1),
        (1, 2)
    ],

    "tfidf__min_df": [
        2,
        3,
        5
    ],

    "tfidf__stop_words": [
        None,
        "english"
    ],

    "classifier__C": [
        0.1,
        0.5,
        1,
        2,
        5
    ]
}

In [ ]:
tfidf_only_search = GridSearchCV(
    estimator=tfidf_only_pipeline,
    param_grid=tfidf_only_param_grid,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
    verbose=2
)

tfidf_only_search.fit(
    X_train["Text"],
    y_train
)

tfidf_only_predictions = tfidf_only_search.predict(
    X_test["Text"]
)

## roberta, moral w/o tfidf

In [ ]:
semantic_only_pipeline = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),

    (
        "classifier",
        LogisticRegression(
            solver="liblinear",
            class_weight="balanced",
            max_iter=3000,
            random_state=RANDOM_STATE
        )
    )
])

In [ ]:
semantic_only_search = GridSearchCV(
    estimator=semantic_only_pipeline,
    param_grid={
        "classifier__C": [
            0.01,
            0.1,
            0.5,
            1,
            2,
            5,
            10
        ]
    },
    scoring="f1_macro",
    cv=5,
    n_jobs=-1
)

semantic_only_search.fit(
    X_train[semantic_features],
    y_train
)

semantic_only_predictions = (
    semantic_only_search.predict(
        X_test[semantic_features]
    )
)

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "TF-IDF only",
        "Sentiment + moral + RoBERTa embeddings only",
        "TF-IDF + sentiment + moral + RoBERTa embeddings"
    ],

    "Accuracy": [
        accuracy_score(
            y_test,
            tfidf_only_predictions
        ),

        accuracy_score(
            y_test,
            semantic_only_predictions
        ),

        accuracy_score(
            y_test,
            full_predictions
        )
    ],

    "Macro F1": [
        f1_score(
            y_test,
            tfidf_only_predictions,
            average="macro"
        ),

        f1_score(
            y_test,
            semantic_only_predictions,
            average="macro"
        ),

        f1_score(
            y_test,
            full_predictions,
            average="macro"
        )
    ],

    "Best CV Macro F1": [
        tfidf_only_search.best_score_,
        semantic_only_search.best_score_,
        full_search.best_score_
    ]
})

comparison

## Semantic feature-block analysis

Individual RoBERTa dimensions are latent and should not be interpreted one by one.  
This section summarizes the overall coefficient magnitude assigned to sentiment, moral-foundation, and embedding feature blocks.


In [ ]:
# Analyze coefficient magnitude by semantic feature block
semantic_classifier = semantic_only_search.best_estimator_.named_steps["classifier"]
semantic_coefficients = semantic_classifier.coef_[0]

n_sentiment = len(sentiment_features)
n_moral = len(moral_features)
n_embeddings = len(roberta_embedding_features)

sentiment_slice = semantic_coefficients[:n_sentiment]
moral_slice = semantic_coefficients[
    n_sentiment:n_sentiment + n_moral
]
embedding_slice = semantic_coefficients[
    n_sentiment + n_moral:
]

semantic_block_summary = pd.DataFrame({
    "Feature block": [
        "RoBERTa sentiment",
        "Moral foundations",
        "RoBERTa embeddings"
    ],
    "Number of features": [
        n_sentiment,
        n_moral,
        n_embeddings
    ],
    "L1 coefficient magnitude": [
        np.abs(sentiment_slice).sum(),
        np.abs(moral_slice).sum(),
        np.abs(embedding_slice).sum()
    ],
    "L2 coefficient magnitude": [
        np.linalg.norm(sentiment_slice),
        np.linalg.norm(moral_slice),
        np.linalg.norm(embedding_slice)
    ],
    "Mean absolute coefficient": [
        np.abs(sentiment_slice).mean(),
        np.abs(moral_slice).mean(),
        np.abs(embedding_slice).mean()
    ]
})

semantic_block_summary


In [ ]:
full_classifier = (
    best_full_model
    .named_steps["classifier"]
)

print(full_classifier.classes_)

In [ ]:
full_preprocessor_fitted = (
    best_full_model
    .named_steps["preprocessor"]
)

feature_names = (
    full_preprocessor_fitted
    .get_feature_names_out()
)

coefficients = full_classifier.coef_[0]

coefficient_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "absolute_coefficient": np.abs(coefficients)
})

In [ ]:
semantic_coefficient_df = (
    coefficient_df[
        coefficient_df["feature"]
        .str.startswith("semantic__")
    ]
    .copy()
)

semantic_coefficient_df["feature"] = (
    semantic_coefficient_df["feature"]
    .str.replace(
        "semantic__",
        "",
        regex=False
    )
)

semantic_coefficient_df = (
    semantic_coefficient_df
    .sort_values(
        "coefficient",
        ascending=False
    )
)

semantic_coefficient_df

In [ ]:
plot_df = semantic_coefficient_df.sort_values(
    "coefficient"
)

plt.figure(figsize=(9, 6))

plt.barh(
    plot_df["feature"],
    plot_df["coefficient"]
)

plt.axvline(
    0,
    linewidth=1
)

plt.xlabel("Logistic regression coefficient")
plt.ylabel("Semantic feature")
plt.title(
    "Sentiment and Moral-Foundation Coefficients"
)

plt.tight_layout()
plt.show()

In [ ]:
semantic_group_summary = (
    model_df
    .groupby("Political Lean")[
        semantic_features
    ]
    .mean()
    .T
)

semantic_group_summary


In [ ]:
semantic_group_summary["Liberal_minus_Conservative"] = (
    semantic_group_summary["Liberal"]
    - semantic_group_summary["Conservative"]
)

semantic_group_summary.sort_values(
    "Liberal_minus_Conservative",
    ascending=False
)

In [ ]:
moral_group_means = (
    model_df
    .groupby("Political Lean")[
        moral_features
    ]
    .mean()
    .T
)

moral_group_means.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.ylabel("Mean semantic similarity")
plt.xlabel("Moral-foundation feature")
plt.title(
    "Average Moral-Foundation Scores by Political Lean"
)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
df

In [ ]:
df.describe()

In [ ]:
df.hist()

In [ ]:
df.plot()